<a href="https://colab.research.google.com/github/jygheo/Contrastive-Decoding/blob/main/text_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')

In [ ]:
import torch
import gc
import torch.nn.functional as F
from transformers import LogitsProcessor, AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList, set_seed
from datasets import load_dataset
import random
import os
import json
import time

set_seed(42)

In [ ]:
def extract_prompt_and_reference(text, tokenizer, prompt_word_count=32, target_token_length=256, skip_first_n_words=0):
    """
    Extracts a word-based prompt (as text) and a human reference (as token IDs).
    """
    words = text.split()

    if len(words) <= skip_first_n_words:
        return None, None

    words = words[skip_first_n_words:]

    if len(words) < (prompt_word_count + 200):
        return None, None

    prompt_words = words[:prompt_word_count]
    prompt_text = " ".join(prompt_words)

    buffer_words = words[prompt_word_count : prompt_word_count + 500]
    remainder_text = " ".join(buffer_words)

    remainder_tokens = tokenizer(remainder_text, add_special_tokens=False, return_tensors="pt").input_ids[0]

    if len(remainder_tokens) < target_token_length:
        return None, None

    reference_tokens = remainder_tokens[:target_token_length]

    return prompt_text, reference_tokens.tolist()

In [ ]:
def get_random_samples(dataset_path, dataset_name=None, split="train", target_count=1000, tokenizer=None, skip_first_n_words=0):
    print(f"Downloading {dataset_path} ({split})")

    ds = load_dataset(dataset_path, dataset_name, split=split, streaming=True)

    ds = ds.shuffle(seed=42)

    valid_samples = []

    for row in ds:
        raw_text = row.get('text')
        prompt, reference = extract_prompt_and_reference(raw_text, tokenizer, skip_first_n_words=skip_first_n_words)

        if prompt and reference:
            valid_samples.append({
                "prompt": prompt,
                "reference": reference
            })

        if len(valid_samples) >= target_count:
            print(f"Target of {target_count} reached")
            break

    del ds

    print(f"Finished extracting {len(valid_samples)} random samples")
    return valid_samples

In [ ]:
def get_random_samples_external(drive_path, target_count=1000, tokenizer=None, skip_first_n_words=1000):
    print(f"Reading files from  directory: {drive_path}")
    file_paths = []
    for root, _, files in os.walk(drive_path):
        for file in files:
            if file.endswith('.txt'):
                file_paths.append(os.path.join(root, file))

    if not file_paths:
        print("No .txt files found ")
        return []

    random.seed(42)
    random.shuffle(file_paths)

    valid_samples = []

    for file_path in file_paths:
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                raw_text = f.read()

            prompt, reference = extract_prompt_and_reference(
                raw_text,
                tokenizer,
                skip_first_n_words=skip_first_n_words
            )

            if prompt and reference:
                valid_samples.append({
                    "prompt": prompt,
                    "reference": reference,
                })

            if len(valid_samples) >= target_count:
                print(f"Target of {target_count} reached")
                break

        except Exception as e:
            print(f"Skipping {file_path} due to error: {e}")
            continue

    print(f"Finished extracting {len(valid_samples)} random samples")
    return valid_samples

In [ ]:
class ContrastiveDecodingLogitsProcessor(LogitsProcessor):
    def __init__(self, amateur_model, alpha=0.1, amateur_temperature=1.0, amateur_context=False):
        self.amateur_model = amateur_model
        self.alpha = alpha
        self.amateur_temperature = amateur_temperature
        self.amateur_context = amateur_context
        self.reset()
    def reset(self):
        #clear cache from previous prompt
        self.past_key_values = None
        self.last_input_ids = None #what beams looked like last step

    def __call__(self, input_ids: torch.LongTensor, expert_logits: torch.FloatTensor) -> torch.FloatTensor:
        original_dtype = expert_logits.dtype

        expert_logits_fp32 = expert_logits.to(torch.float32)
        expert_probs = F.softmax(expert_logits_fp32, dim=-1)

        with torch.no_grad():
            if self.past_key_values is None:
                if self.amateur_context:
                    amateur_outputs = self.amateur_model(input_ids, use_cache=True)
                else:
                    amateur_outputs = self.amateur_model(input_ids[:, -1:], use_cache=True)
            else:
                #compare cur beams w/o newest token vs cached (prev) beams to see which parent each came from
                matches = (input_ids[:, :-1].unsqueeze(1) == self.last_input_ids.unsqueeze(0)).all(dim=-1)
                parent_indices = matches.long().argmax(dim=1)

                if hasattr(self.past_key_values, "reorder_cache"):
                    self.past_key_values.reorder_cache(parent_indices)

                elif hasattr(self.amateur_model, "_reorder_cache"):
                    self.past_key_values = self.amateur_model._reorder_cache(self.past_key_values, parent_indices)

                elif isinstance(self.past_key_values, tuple):
                    self.past_key_values = tuple(
                        tuple(past_state.index_select(0, parent_indices) for past_state in layer_past)
                        for layer_past in self.past_key_values
                    )
                else:
                    raise ValueError(f"Unrecognized cache type: {type(self.past_key_values)}")

                amateur_outputs = self.amateur_model(
                    input_ids[:, -1:],
                    past_key_values=self.past_key_values,
                    use_cache=True
                )
            self.past_key_values = amateur_outputs.past_key_values
            self.last_input_ids = input_ids

            amateur_logits = amateur_outputs.logits[:, -1, :].to(torch.float32)
            amateur_logits = amateur_logits / self.amateur_temperature
            amateur_probs = F.softmax(amateur_logits, dim=-1)

        max_expert_probs, _ = torch.max(expert_probs, dim=-1, keepdim=True)
        plausibility_mask = expert_probs >= (self.alpha * max_expert_probs)

        eps = 1e-10
        expert_log_probs = torch.log(expert_probs + eps)
        amateur_log_probs = torch.log(amateur_probs + eps)

        cd_scores = expert_log_probs - amateur_log_probs
        cd_scores = cd_scores.masked_fill(~plausibility_mask, float('-inf'))

        return cd_scores.to(original_dtype)

In [ ]:
EXPERIMENT_CONFIG = {
    "gpt2": {
        "temp": 0.5,
        "models": ["gpt2", "gpt2-medium", "gpt2-large", "gpt2-xl"] # 124M, 355M, 774M, 1.5B
    },
    "opt": {
        "temp": 1.0,
        "models": ["facebook/opt-125m", "facebook/opt-350m", "facebook/opt-1.3b", "facebook/opt-2.7b", "facebook/opt-6.7b"]
    },
    "qwen": {
        "temp": 1.0,
        "models": ["Qwen/Qwen1.5-0.5B", "Qwen/Qwen1.5-1.8B", "Qwen/Qwen1.5-4B", "Qwen/Qwen1.5-7B"]
    }
}

In [ ]:
class ContrastiveExperimentPipeline:
    def __init__(self, expert_id, amateur_id, family, scaling=False, amateur_context=False):
        self.expert_id = expert_id
        self.amateur_id = amateur_id
        self.family = family

        self.temp =  1.0 if scaling else EXPERIMENT_CONFIG[family]["temp"]
        self.amateur_context = amateur_context

        self.tokenizer = AutoTokenizer.from_pretrained(expert_id)
        self.tokenizer.padding_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.expert_model = AutoModelForCausalLM.from_pretrained(
            expert_id,
            device_map="auto",
            dtype=torch.float16,
            attn_implementation="sdpa",
        )
        self.amateur_model = AutoModelForCausalLM.from_pretrained(
            amateur_id,
            device_map="auto",
            dtype=torch.float16,
            attn_implementation="sdpa",
        )

        self.cd_processor = ContrastiveDecodingLogitsProcessor(
            amateur_model=self.amateur_model,
            alpha=0.1,
            amateur_temperature=self.temp,
            amateur_context=self.amateur_context
        )
        self.logits_processor = LogitsProcessorList([self.cd_processor])

    def prepare_batched_prompts(self, prompt_texts):
        truncated_texts = [" ".join(p.split()[:32]) for p in prompt_texts]
        return self.tokenizer(
            truncated_texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.expert_model.device)

    def generate_all_baselines(self, prompt_texts):
        """Performance comparisons across sampling methods experiment"""
        inputs = self.prepare_batched_prompts(prompt_texts)
        input_len = inputs.input_ids.shape[1]

        decoding_strategies = {
            "Greedy": {"do_sample": False},
            "Top-k": {"do_sample": True, "top_k": 50},
            "Nucleus": {"do_sample": True, "top_p": 0.95},
            "Typical": {"do_sample": True, "typical_p": 0.95},
            "CS": {"penalty_alpha": 0.6, "top_k": 5, "trust_remote_code": True},
            "CD_Search": {"do_sample": False, "num_beams": 5, "logits_processor": self.logits_processor}
        }

        results = {name: [] for name in decoding_strategies.keys()}

        for name, kwargs in decoding_strategies.items():
            if "logits_processor" in kwargs:
                self.cd_processor.reset()
            outputs = self.expert_model.generate(
                **inputs,
                min_new_tokens=256,
                max_new_tokens=256,
                max_length=None,
                pad_token_id=self.tokenizer.pad_token_id,
                **kwargs
            )
            continuation_ids = outputs[:, input_len:]
            results[name] = continuation_ids.tolist()
        return results

    def generate_cd_only(self, prompt_texts):
        """Model scaling experiment"""
        inputs = self.prepare_batched_prompts(prompt_texts)
        input_len = inputs.input_ids.shape[1]
        self.cd_processor.reset()

        outputs = self.expert_model.generate(
            **inputs,
            min_new_tokens=256,
            max_new_tokens=256,
            max_length=None,
            pad_token_id=self.tokenizer.pad_token_id,
            do_sample=False,
            num_beams=5,
            logits_processor=self.logits_processor
        )

        continuation_ids = outputs[:, input_len:]
        return continuation_ids.tolist()

    def cleanup(self):
        del self.expert_model
        del self.amateur_model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
def run_experiment(expert_id, amateur_id, family, data, dataset_name, scaling=False, amateur_context=False, batch_size=8):
    """
    Runs the pipeline, saves to Google Drive as a JSONL file.
    cd_only should be true for scaling experiment
    """
    cd_only = False
    if scaling:
        cd_only=True

    pipeline = ContrastiveExperimentPipeline(
        expert_id=expert_id,
        amateur_id=amateur_id,
        family=family,
        scaling=scaling,
        amateur_context=amateur_context
    )

    safe_expert = pipeline.expert_id.split('/')[-1]
    safe_amateur = pipeline.amateur_id.split('/')[-1]
    experiment_folder = "scaling-temp-1" if scaling else "baselines"
    if pipeline.amateur_context:
        experiment_folder += "-amateur-full-context"


    base_path = f"/content/drive/MyDrive/{experiment_folder}/{pipeline.family}/"
    os.makedirs(base_path, exist_ok=True)
    file_path = os.path.join(base_path, f"{safe_expert}_vs_{safe_amateur}_{dataset_name}.jsonl")

    print(f"Saving to: {file_path}")

    with open(file_path, "a", encoding="utf-8") as f:

        for i in range(0, len(data), batch_size):
            batch = data[i : i + batch_size]
            prompts = [pair["prompt"] for pair in batch]
            references = [pair["reference"] for pair in batch]

            if cd_only:
                generations = {"CD_Search": pipeline.generate_cd_only(prompts)}
            else:
                generations = pipeline.generate_all_baselines(prompts)

            for j in range(len(batch)):
                record = {
                    "id": i + j,
                    "prompt": prompts[j],
                    "human_reference": references[j],
                    "generations": {method: text_list[j] for method, text_list in generations.items()}
                }
                f.write(json.dumps(record) + "\n")

            f.flush()
            os.fsync(f.fileno())

            print(f"Saved {min(i + batch_size, len(data))}/{len(data)} samples")

    print(f"Finished {dataset_name}\n")
    pipeline.cleanup()

Comparisons across sampling methods for GPT-2 family

In [ ]:
TARGET_SAMPLES = 1000
tokenizer = AutoTokenizer.from_pretrained("gpt2")

wikinews_samples = get_random_samples(
    dataset_path="izumi-lab/wikinews-en-20230728",
    dataset_name=None,
    split="train",
    target_count=TARGET_SAMPLES,
    tokenizer=tokenizer
)

wikitext_samples = get_random_samples(
    dataset_path="Salesforce/wikitext",
    dataset_name="wikitext-103-raw-v1",
    split="train",
    target_count=TARGET_SAMPLES,
    tokenizer=tokenizer
)


gutenberg_samples = get_random_samples_external(
    drive_path="/content/drive/MyDrive/Gutenberg",
    target_count=TARGET_SAMPLES,
    tokenizer=tokenizer
)


run_experiment("gpt2-xl", "gpt2", "gpt2" wikinews_samples, dataset_name="wikinews", batch_size=16)
run_experiment("gpt2-xl", "gpt2", "gpt2" wikitext_samples, dataset_name="wikitext", batch_size=16)
run_experiment("gpt2-xl", "gpt2", "gpt2" gutenberg_samples, dataset_name="gutenberg", batch_size=16)

Comparisons across sampling methods for OPT family

In [ ]:
opt13b_pipeline = ContrastiveExperimentPipeline(
    expert_id="facebook/opt-6.7b",
    amateur_id="facebook/opt-125m",
    family="opt"
)

TARGET_SAMPLES = 1000
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-6.7b", use_fast=False)

# wikinews_samples = get_random_samples(
#     dataset_path="izumi-lab/wikinews-en-20230728",
#     dataset_name=None,
#     split="train",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )

# wikitext_samples = get_random_samples(
#     dataset_path="Salesforce/wikitext",
#     dataset_name="wikitext-103-raw-v1",
#     split="train",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )


# gutenberg_samples = get_random_samples_external(
#     drive_path="/content/drive/MyDrive/Gutenberg",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )

run_experiment("facebook/opt-6.7b", "facebook/opt-125m", "opt", wikinews_samples, dataset_name="wikinews",  batch_size=8)
run_experiment("facebook/opt-6.7b", "facebook/opt-125m", "opt", wikitext_samples, dataset_name="wikitext",  batch_size=8)
run_experiment("facebook/opt-6.7b", "facebook/opt-125m", "opt", gutenberg_samples, dataset_name="gutenberg",  batch_size=8)

Comparisons across sampling methods for Qwen-1.5 family

In [ ]:
qwen_pipeline = ContrastiveExperimentPipeline(
    expert_id="Qwen/Qwen1.5-7B",
    amateur_id="Qwen/Qwen1.5-0.5B",
    family="qwen"
)

TARGET_SAMPLES = 1000
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B")

# wikinews_samples = get_random_samples(
#     dataset_path="izumi-lab/wikinews-en-20230728",
#     dataset_name=None,
#     split="train",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )

# wikitext_samples = get_random_samples(
#     dataset_path="Salesforce/wikitext",
#     dataset_name="wikitext-103-raw-v1",
#     split="train",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )


# gutenberg_samples = get_random_samples_external(
#     drive_path="/content/drive/MyDrive/Gutenberg",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )

run_experiment("Qwen/Qwen1.5-7B", "Qwen/Qwen1.5-0.5B", "qwen", wikinews_samples, dataset_name="wikinews",  batch_size=16)
run_experiment("Qwen/Qwen1.5-7B", "Qwen/Qwen1.5-0.5B", "qwen", wikitext_samples, dataset_name="wikitext", batch_size=16)
run_experiment("Qwen/Qwen1.5-7B", "Qwen/Qwen1.5-0.5B", "qwen", gutenberg_samples, dataset_name="gutenberg",batch_size=16)


Scaling experiment for GPT family

In [ ]:
family = "gpt2"
TARGET_SAMPLES = 1000
tokenizer = AutoTokenizer.from_pretrained("gpt2")

models_to_test = EXPERIMENT_CONFIG[family]["models"]

# wikitext_samples = get_random_samples(
#     dataset_path="Salesforce/wikitext",
#     dataset_name="wikitext-103-raw-v1",
#     split="train",
#     target_count=TARGET_SAMPLES,
#     tokenizer=tokenizer
# )


run_experiment("gpt2", "gpt2-medium", "gpt2", wikitext_samples, dataset_name="wikitext",  batch_size=32, scaling=True)

for expert_id in models_to_test[::-1]:
    for amateur_id in models_to_test[::-1]:
        run_experiment(expert_id, amateur_id, family, wikitext_samples, dataset_name="wikitext",  batch_size=32, scaling=True)